# S02 · 01 — El dolor de los datos: todo verde, todo mal

**Este es "el dolor" de la sesión 2.** No se abre Pandera hasta la sección 3.

**Objetivo.** Cambiar **una sola cosa** en el dato de entrada —`trip_distance` de
millas a kilómetros— y comprobar que el `pipeline` **no falla**: entrena, reporta un
RMSE perfectamente creíble y sirve predicciones. Después, ver qué tipo de
comprobación sí lo atrapa, y **cuál es su límite honesto**.

**Por qué importa.** Los errores de datos casi nunca lanzan excepciones. Degradan
métricas. Un `pipeline` que se cae es un problema de una tarde; un `pipeline` que
sigue funcionando con datos que cambiaron de significado es un problema que se
descubre meses después, cuando alguien pregunta por qué las predicciones no tienen
sentido.

**Requisito.** Nada. Este notebook usa los `fixtures` sintéticos deterministas de
[`tests/conftest.py`](../../../tests/conftest.py), así que **no necesita red**. La
sección 5 sí usa el parquet real: si no lo tienes, corre `make data` o salta esa
sección (el notebook lo detecta y avisa).

**Ruta del notebook.**

1. El fallo silencioso: entrenar en millas y en "km", y comparar las dos métricas.
2. Segundo acto: una categoría nueva que el `encoder` descarta sin decir nada.
3. Los tres niveles de check, y cuál atrapa cada cosa.
4. Qué va en el contrato y qué va en el test.
5. **El límite honesto**: el mismo cambio de unidades sobre el parquet **real**
   pasa el contrato. Y qué sí lo detecta (puente a la S07).

In [ ]:
# Preambulo: hacer importables `taxi` (src/) y `tests.conftest` (los fixtures).
# En un notebook del proyecto propio esto no hace falta si el paquete esta
# instalado con `uv sync` (que es lo correcto); aqui se hace explicito para que el
# notebook funcione tambien sin instalar nada.
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "pyproject.toml").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
for p in (str(RAIZ), str(RAIZ / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("raiz:", RAIZ.name)

---

## 1. El fallo silencioso: millas contra kilómetros

Los `fixtures` de este notebook son los mismos que usa la suite de tests del
repositorio. Se **generan en código** con semilla fija, en lugar de leerse de un CSV
commiteado, por tres razones que están escritas en
[`tests/conftest.py`](../../../tests/conftest.py):

- un CSV de mil filas es **opaco**: nadie sabe qué propiedad tiene;
- se desactualiza en silencio cuando el contrato cambia;
- no explica **por qué** cada `fixture` roto está roto.

Generarlo con semilla fija da reproducibilidad bit a bit *y* documenta la intención.

In [ ]:
import pandas as pd

from tests.conftest import MILLAS_A_KM, generar_crudos

# El "proveedor" nos manda datos correctos, en MILLAS (es lo que documenta la NYC TLC).
crudo_millas = generar_crudos(filas=4_000, semilla=11)

# Y un dia, sin avisar, cambia de unidad. UNA columna. Nada mas.
crudo_km = crudo_millas.copy()
crudo_km["trip_distance"] = crudo_millas["trip_distance"] * MILLAS_A_KM

comparacion = pd.DataFrame(
    {
        "millas": crudo_millas["trip_distance"].describe(),
        "km (el dato roto)": crudo_km["trip_distance"].describe(),
    }
)
print(comparacion.round(2))
print()
print("dtypes iguales:", crudo_millas.dtypes.equals(crudo_km.dtypes))
print("nulos nuevos:", int(crudo_km.isna().sum().sum() - crudo_millas.isna().sum().sum()))
print("filas iguales:", len(crudo_millas) == len(crudo_km))

**Mira la tabla antes de seguir.** El tipo no cambió. No apareció ningún nulo. El
número de filas es el mismo. La mediana pasó de ~3,4 a ~5,5 millas y **las dos son
plausibles para un taxi**.

No hay nada que un `try/except` pueda atrapar. No hay nada que `mypy` pueda ver: el
tipo sigue siendo `float64`. Un `float` no lleva sus unidades encima.

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

from taxi.config import DURACION_MAX_MIN, DURACION_MIN_MIN
from taxi.features import contract as fc


def preparar(df: pd.DataFrame) -> pd.DataFrame:
    # El mismo orden que taxi.data.loaders.preparar_particion, sin descargar nada.
    df = df.copy()
    delta = df[fc.COL_DROPOFF] - df[fc.COL_PICKUP]
    df[fc.TARGET_REGRESION] = delta.dt.total_seconds() / 60.0
    dentro = df[fc.TARGET_REGRESION].between(DURACION_MIN_MIN, DURACION_MAX_MIN)
    return fc.construir_features(df[dentro].reset_index(drop=True))


def entrenar_y_medir(train: pd.DataFrame, test: pd.DataFrame) -> float:
    dv = DictVectorizer()
    x_tr = dv.fit_transform(fc.a_diccionarios(train))
    x_te = dv.transform(fc.a_diccionarios(test))
    modelo = Ridge(alpha=1.0).fit(x_tr, train[fc.TARGET_REGRESION])
    return float(root_mean_squared_error(test[fc.TARGET_REGRESION], modelo.predict(x_te)))


# El conjunto de evaluacion llega SIEMPRE en millas: es el mundo real, que no cambio.
test = preparar(generar_crudos(filas=1_500, semilla=99))

rmse_ok = entrenar_y_medir(preparar(crudo_millas), test)
rmse_roto = entrenar_y_medir(preparar(crudo_km), test)

print(f"RMSE del modelo entrenado en millas : {rmse_ok:6.3f} min")
print(f"RMSE del modelo entrenado en 'km'   : {rmse_roto:6.3f} min")
print()
print(f"degradacion: {100 * (rmse_roto - rmse_ok) / rmse_ok:.1f}%")
print()
print("Ninguna excepcion. Ningun aviso. Dos numeros creibles.")

### Lo que acaba de pasar, y por qué es el peor caso posible

El modelo entrenado con la columna equivocada **funciona**. Predice minutos. Su RMSE
es del mismo orden que el del modelo correcto. Si lo registras en MLflow, la métrica
se ve bien. Si lo despliegas, la API responde `200` en 8 ms.

Y es un modelo que aprendió que "3,4 unidades de distancia" significa una cosa,
cuando en producción va a recibir otra.

**Mide y compara** la degradación que te salga. Después piensa en la pregunta
incómoda: *¿cuánta degradación haría falta para que alguien lo notase sin un
contrato?* Con una diferencia de un 10 % y sin nada con lo que comparar, la respuesta
honesta es que probablemente nadie.

> **La asimetría que define esta sesión.** Un fallo ruidoso cuesta una tarde de
> `debugging`. Un fallo silencioso cuesta meses de decisiones tomadas con
> predicciones malas, más el tiempo de descubrir que el problema estaba en los datos
> y no en el modelo. El contrato de datos convierte lo segundo en lo primero.

---

## 2. Segundo acto: la categoría nueva que desaparece

El cambio de unidades es el caso canónico, pero hay uno todavía más silencioso, y le
pasa a todo el mundo: **una categoría que no existía cuando entrenaste**.

El caso guía usa `DictVectorizer` para las features categóricas, y lo hace a
propósito (está explicado en
[`src/taxi/features/contract.py`](../../../src/taxi/features/contract.py)): el par
origen-destino `PU_DO` tiene miles de valores posibles y varios aparecen **solo** en
producción.

`DictVectorizer` ignora las claves que no vio en `fit`. Eso es el comportamiento
deseado —no queremos que la API explote porque llegó una ruta nueva— pero tiene un
precio que hay que ver con los ojos.

In [ ]:
dv = DictVectorizer()
train = preparar(generar_crudos(filas=2_000, semilla=3))
dv.fit(fc.a_diccionarios(train))
print("features aprendidas en fit:", len(dv.get_feature_names_out()))

# Un solo viaje, con una zona que NUNCA aparecio en entrenamiento.
viaje_nuevo = {
    "PU_DO": "999_999",
    "PULocationID": "999",
    "DOLocationID": "999",
    "trip_distance": 4.2,
    "hora_pickup": 9,
    "dia_semana_pickup": 2,
}
fila = dv.transform([viaje_nuevo])

nombres = dv.get_feature_names_out()
activas = [(nombres[j], float(fila[0, j])) for j in fila.nonzero()[1]]
print()
print("valores no cero de la fila transformada:", activas)
print()
print("Ni excepcion, ni warning. Las tres features categoricas valen 0.")
print("El modelo predice usando SOLO la distancia y la hora, y no lo dice.")

### La pregunta del segundo acto

El modelo va a devolver un número. Va a ser un número razonable. Y va a estar
calculado ignorando el 50 % de las features del viaje.

**¿Cuántas predicciones así puede estar sirviendo tu API ahora mismo?** Sin
instrumentación, la respuesta es que no lo sabes. Y hay tres formas distintas de
enterarse, cada una con su sitio en el curso:

| Cómo | Cuándo se enteras | Sesión |
|---|---|---|
| El **contrato** rechaza el lote si la zona está fuera de `1-265` | en la frontera, antes de entrenar | esta (§3) |
| Una **métrica** que cuenta categorías no vistas por `request` | en producción, en tiempo real | S07 (Prometheus) |
| El **drift** de la distribución de `PU_DO` contra la referencia | por lotes, comparando periodos | S07 (Evidently) |

Las tres, no una. Un contrato valida **la forma** del dato que entra; no puede
saber que la zona 42 dejó de existir en el mundo.

**Nota sobre alternativas:** con `OneHotEncoder(handle_unknown="ignore")` pasa
exactamente lo mismo (por eso existe ese parámetro). Con
`handle_unknown="error"` la API se cae en la primera ruta nueva. Ninguna de las dos
opciones es gratis: o degradas en silencio, o rompes en voz alta. Lo que no vale es
no haber elegido.

---

## 3. Los tres niveles de check

**Ahora sí** abrimos el contrato. Está implementado y documentado en
[`src/taxi/data/contract.py`](../../../src/taxi/data/contract.py); este notebook
**ejercita** lo que ya está ahí.

La clase `ViajesCrudos` distingue tres niveles a propósito, porque responden
preguntas distintas:

| Nivel | Cómo se escribe | Qué atrapa | Ejemplo del contrato |
|---|---|---|---|
| **1. Por fila** | `pa.Field(ge=..., le=...)` | registros corruptos individuales | `PULocationID` en `1-265`, `trip_distance >= 0` |
| **2. Por distribución** | `@pa.dataframe_check` sobre una **fracción** o un agregado | problemas **sistemáticos**: un cambio de unidades, una ingesta cortada | `outliers_de_distancia_son_marginales`, `volumen_minimo` |
| **3. Entre columnas** | `@pa.dataframe_check` con dos o más columnas | incoherencias que no se ven mirando una columna sola | `velocidad_implicita_plausible`, `dropoff_posterior_a_pickup` |

Y la decisión de diseño que más cuesta aceptar: **la cota del nivel 1 es ancha a
propósito.**

In [ ]:
import pandera.errors as pae

from taxi.data import contract as dc


def diagnosticar(df: pd.DataFrame, etiqueta: str) -> None:
    # Devuelve QUE check fallo, no solo que fallo. Es la diferencia entre un
    # contrato que ensena y uno que solo bloquea.
    try:
        dc.validar_crudos(df)
        print(f"{etiqueta:42s} PASA")
    except pae.SchemaErrors as exc:
        checks = sorted({str(c) for c in exc.failure_cases["check"]})
        print(f"{etiqueta:42s} FALLA -> {checks}")
    except pae.SchemaError as exc:
        print(f"{etiqueta:42s} FALLA -> {exc.check}")


base = generar_crudos(filas=2_000, semilla=7)

diagnosticar(base, "crudo valido (control positivo)")
print()

print("--- Nivel 1: por fila ---")
d = base.copy()
d.loc[d.index[0], "trip_distance"] = -3.0
diagnosticar(d, "distancia negativa (imposible)")

d = base.copy()
d.loc[d.index[:3], "DOLocationID"] = 999
diagnosticar(d, "zona 999 (fuera del rango 1-265 de la TLC)")

d = base.copy()
d.loc[d.index[:1], "trip_distance"] = 120_098.84
diagnosticar(d, "1 outlier de 120.098 millas")

print()
print("--- Nivel 2: por distribucion ---")
d = base.copy()
cuantas = int(len(d) * 0.02)
d.loc[d.index[:cuantas], "trip_distance"] = 150.0
diagnosticar(d, "el 2% supera las 100 millas")

d = base.copy()
d["trip_distance"] = d["trip_distance"] * MILLAS_A_KM
diagnosticar(d, "millas -> km (el fallo de la seccion 1)")

diagnosticar(base.head(50), "solo 50 filas (ingesta cortada)")

print()
print("--- Nivel 3: entre columnas ---")
d = base.copy()
d[fc.COL_DROPOFF] = d[fc.COL_PICKUP] + pd.Timedelta(seconds=20)
diagnosticar(d, "velocidad implicita de ~300 mph")

d = base.copy()
d.loc[d.index[:2], fc.COL_DROPOFF] = d.loc[d.index[:2], fc.COL_PICKUP] - pd.Timedelta(minutes=5)
diagnosticar(d, "el viaje termina antes de empezar")

### La fila que hay que mirar dos veces

> `1 outlier de 120.098 millas` → **PASA**

Esto no es un descuido: es la decisión más importante del contrato, y viene de un bug
real. El contrato original exigía `trip_distance <= 100` **por fila**, y el parquet
real de la TLC de 2023-01 trae **37 viajes de más de 100 millas** (uno de 120.098) en
**68.211** filas. Resultado: `taxi data` fallaba con `SchemaErrors` en la primera
partición y bloqueaba el curso entero.

Un contrato que rechaza el lote completo por 37 filas de 68.211 es un contrato que el
equipo **aprende a desactivar**, y un contrato desactivado protege exactamente cero.

La cota por fila se queda en lo que ninguna fila válida puede violar (no negativa, no
absurda por órdenes de magnitud) y **la señal de "esto es sistemático" se mueve al
nivel 2**, que es donde pertenece:

```python
# 0,054% de los viajes de 2023-01 supera las 100 millas (medido).
# El umbral de 0,3% deja 5x de margen y sigue atrapando una inflacion sistematica.
MAX_FRACCION_OUTLIERS: float = 0.003
```

**La regla que hay que llevarse:** *un puñado de registros absurdos es ruido de
captura y se filtra; que el 1 % de los viajes pase de 100 millas significa que la
columna cambió de significado.* Son dos preguntas distintas y necesitan dos checks
distintos.

Los tests que fijan esto están en
[`tests/data/test_niveles_de_check.py`](../../../tests/data/test_niveles_de_check.py),
incluido uno que protege el umbral de que alguien lo baje por debajo del ruido real.

In [ ]:
# El filtro NO es silencioso, y eso tambien es una decision.
# `preparar_particion` cuenta cuantas filas descarta y avisa si pasa del 35%.
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from taxi.data import loaders  # noqa: E402

print(
    "Lee el log de loaders.preparar_particion: registra CUANTAS filas descarto y",
    "avisa si descarta mas del 35%.",
)
print()
print("Por que importa: un filtro silencioso es tan peligroso como no tener filtro.")
print("Si manana se descarta el 40% de los datos, alguien tiene que enterarse.")
print()
print("Fuente:", Path(loaders.__file__).relative_to(RAIZ))

---

## 4. Qué va en el contrato y qué va en el test

Se confunden constantemente, y la distinción es simple:

- **El contrato describe el dato válido.** Es una afirmación sobre el mundo:
  *"`PULocationID` está entre 1 y 265"*. Vive en `src/`, se versiona con el código y
  se ejecuta **en la frontera** del `pipeline`: donde el dato entra, no donde se usa.
- **El test verifica cómo se comporta el pipeline ante el dato inválido.** Es una
  afirmación sobre tu código: *"si llega la zona 999, la validación **falla**"*. Vive
  en `tests/`, se ejecuta en el CI y **nunca** en producción.

Dicho de otra forma: el contrato es la regla, el test es la prueba de que la regla
está encendida.

**Por qué hacen falta los dos.** Un contrato sin tests se degrada: basta que alguien
relaje un rango para desbloquear un `pipeline` un viernes por la tarde, y nadie lo
nota. Con tests, esa relajación rompe un test con nombre explícito. Es literalmente
lo que dice el encabezado de
[`tests/data/test_contrato_datos.py`](../../../tests/data/test_contrato_datos.py).

In [ ]:
# El control negativo, que es el test que casi nadie escribe:
# el contrato NO debe inventar fallos.
for semilla in (1, 2, 3, 4, 5):
    dc.validar_crudos(generar_crudos(semilla=semilla))
print("5 lotes independientes del fixture valido: los tres niveles pasan.")
print()
print("Un contrato que falla con datos buenos se desactiva en la primera semana.")
print("Sin este test, 'mi contrato detecta el fallo' no demuestra nada:")
print("un contrato que rechaza TODO tambien detecta el fallo.")

---

## 5. El límite honesto, y el puente a la sesión 7

Aquí es donde este notebook se separa de la mayoría del material que hay por ahí.
Todo lo anterior funcionó sobre un `fixture` **sintético**. Vamos a repetir el
experimento sobre el parquet **real** de la TLC.

Aviso de resultado, para que la sorpresa sea la correcta: **el contrato va a pasar**.

In [ ]:
from taxi.config import RAW_DIR

PARQUET = RAW_DIR / "green_tripdata_2023-01.parquet"
print("parquet real presente:", PARQUET.exists())
if not PARQUET.exists():
    print()
    print("Corre `make data` (o `uv run taxi data`) para esta seccion.")
    print("Sin el parquet, salta a la seccion 6.")

In [ ]:
real = pd.read_parquet(PARQUET)
real_km = real.copy()
real_km["trip_distance"] = real["trip_distance"] * MILLAS_A_KM

print(f"filas: {len(real):,}")
print(f"mediana en millas: {real['trip_distance'].median():.2f}")
print(f"mediana en 'km'  : {real_km['trip_distance'].median():.2f}")
print()
frac_millas = float((real["trip_distance"] > 100).mean())
frac_km = float((real_km["trip_distance"] > 100).mean())
print(f"fraccion > 100 en millas: {frac_millas:.6f}  ({int(frac_millas * len(real))} viajes)")
print(f"fraccion > 100 en 'km'  : {frac_km:.6f}  ({int(frac_km * len(real))} viajes)")
print(f"umbral del contrato     : {dc.MAX_FRACCION_OUTLIERS}")
print()
diagnosticar(real, "parquet REAL en millas")
diagnosticar(real_km, "parquet REAL en 'km'")

### Por qué pasa, y por qué hay que decirlo en voz alta

El contrato **no atrapa** el cambio de unidades sobre el lote real. La razón es
exacta y está escrita en
[`tests/data/test_contrato_datos.py`](../../../tests/data/test_contrato_datos.py):

> *"El contrato no compara unidades (no puede: el número no las lleva). Lo que
> detecta es que los viajes largos legítimos, al convertirse a km, se salen del
> límite de 100. **Si el dataset no tuviera cola larga, este contrato NO atraparía el
> cambio de unidades**, y conviene que eso quede escrito."*

El `fixture` sintético tiene una cola diseñada de viajes de 70-95 millas, así que en
km se sale de rango y el check de nivel 2 se dispara. El parquet real de green taxi
es abrumadoramente urbano: su mediana es 1,85 millas y su p99 no llega a 15. Al pasar
a km, la mediana va a ~2,98 y **sigue siendo perfectamente plausible para un taxi**.

**Un factor de 1,6 está en el borde de lo que un contrato estático puede detectar
sobre un solo lote.** Eso no es un defecto del contrato: es su definición. Un
contrato mira **un** lote, aislado, y decide si está bien formado. Para detectar un
cambio de escala moderado hace falta lo único que el contrato no tiene: **una
referencia con la que comparar**.

Y eso es monitoreo de `drift`, que es la sesión 7.

In [ ]:
# Lo que SI detecta el cambio de unidades: comparar contra una referencia.
# Este es literalmente el primer instrumento de la sesion 7.
from scipy.stats import ks_2samp

referencia = real["trip_distance"].sample(20_000, random_state=1)
actual = real_km["trip_distance"].sample(20_000, random_state=2)

r = ks_2samp(referencia, actual)
print("KS entre la referencia (millas) y el lote actual (km):")
print(f"  D = {r.statistic:.4f}   <- tamano de efecto, acotado en [0, 1]")
print(f"  p = {r.pvalue:.3g}")
print()
print("Un D de ese orden es enorme. El contrato no lo vio; el drift no tiene dudas.")
print()
print("Y el control negativo, que es lo que hace creible al detector:")
mitad_a = real["trip_distance"].sample(20_000, random_state=10)
mitad_b = real["trip_distance"].sample(20_000, random_state=20)
r0 = ks_2samp(mitad_a, mitad_b)
print(f"  dos muestras del MISMO dato: D = {r0.statistic:.4f}  (ruido del instrumento)")

### El puente S02 → S07, que es lo que hay que llevarse de este notebook

| | Contrato de datos (S02) | Monitoreo de `drift` (S07) |
|---|---|---|
| Qué mira | **un** lote, aislado | **dos** distribuciones: referencia vs actual |
| Qué pregunta responde | ¿este dato está bien formado? | ¿este dato se parece al que usé para entrenar? |
| Cuándo actúa | en la **frontera**, antes de entrenar o servir | por lotes, después, de forma continua |
| Qué hace al fallar | detiene el `pipeline` (falla rápido y ruidoso) | alerta, y alguien **investiga** |
| Qué **no** puede ver | un cambio de escala moderado, una categoría que dejó de existir en el mundo | un registro individual corrupto |

**No compiten. Se complementan, y un curso que enseñe solo una de las dos deja
justamente el hueco por el que se cuelan los incidentes reales.**

Si te quedas solo con el contrato, un factor de 1,6 se te pasa. Si te quedas solo con
el `drift`, entrenas con nulos y con zonas que no existen mientras esperas a que se
acumule un lote para comparar.

---

## 6. Autoverificación del notebook

1. El cambio de unidades no lanzó ninguna excepción y `mypy` habría pasado en verde.
   ¿Por qué? ¿Qué información falta en un `float64`?
2. ¿Por qué el contrato **acepta** un viaje de 120.098 millas y **rechaza** que el
   2 % de los viajes pase de 100? Explícalo en términos de las dos preguntas
   distintas que responden.
3. Bajas `MAX_FRACCION_OUTLIERS` de `0.003` a `0.0001`. ¿Qué gana el contrato y qué
   pierde? (Pista: mide la fracción real de 2023-01, que salió en la §5.)
4. El contrato pasó sobre el parquet real en km. Nombra **dos** cosas distintas que sí
   lo habrían detectado, y di en qué momento del ciclo de vida actúa cada una.
5. Tu API recibe una zona que no existía en entrenamiento. ¿Devuelve un error o una
   predicción? ¿Cuál de las dos opciones quieres, y de qué depende la respuesta?

## 7. Ejercicios (10 min)

1. Cambia el factor de conversión de `1.60934` a `1.15` (millas a millas náuticas) y
   vuelve a correr la §3. ¿A partir de qué factor el contrato deja de detectarlo con
   el `fixture` sintético? **Mide el umbral**, no lo estimes.
2. Escribe un check de **nivel 3** nuevo: que la relación entre `total_amount` y
   `trip_distance` sea plausible. Piensa el rango antes de mirar los datos, y después
   mídelo. ¿Coincidió?
3. Rompe un check a propósito en `src/taxi/data/contract.py` (relaja un rango) y corre
   `uv run pytest tests/data -q`. ¿Qué test se cae y qué dice su nombre?
   **Deshaz el cambio.**
4. Añade un cuarto `fixture` roto a `tests/conftest.py` —un `timestamp` en el
   futuro— y el test que verifica que el contrato lo rechaza. Si el contrato **no** lo
   rechaza, has encontrado un hueco: escríbelo.

Siguiente: [`02-validacion-temporal-y-leakage.ipynb`](02-validacion-temporal-y-leakage.ipynb) ·
Volver: [README de la sesión](../README.md)